# Hyperparameter Sweep: Vanilla RNN vs LSTM

Systematic comparison of model architectures and hyperparameters across both
**behavior prediction** (choice) and **neural prediction** (PSTH) tasks.

Results are logged to `results/sweep_results.csv` and visualized with
architecture-agnostic plots.

---

### Running on GCP

1. **Clone the repo** onto your VM:
   ```bash
   git clone https://github.com/stellinaao/df-rnns.git && cd df-rnns
   ```
2. **Create a conda env** and install dependencies:
   ```bash
   conda create -n dfrnns python=3.12 -y && conda activate dfrnns
   pip install -r requirements-gcp.txt
   ```
3. **Upload data** (scp or gcloud):
   ```bash
   # Copy the NP analysis data folder to the VM
   scp -r /path/to/DynamicForagingNPanalysis/data user@<VM_IP>:~/data
   ```
4. **Set the `DATA_ROOT` env var** before launching the notebook:
   ```bash
   export DATA_ROOT=~/data
   ```
5. **Launch JupyterLab** (or use VS Code Remote-SSH):
   ```bash
   jupyter lab --no-browser --port=8888
   ```
   Then SSH-tunnel to your VM: `ssh -L 8888:localhost:8888 user@<VM_IP>`

6. **Use tmux** so long-running sweeps survive SSH disconnects:
   ```bash
   tmux new -s sweep
   jupyter lab --no-browser --port=8888
   # Ctrl-b d to detach; tmux a -t sweep to re-attach
   ```

## 1. Imports and Setup

In [1]:
import os
import random
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# Add repo/src to sys.path
try:
    _nb_dir = Path(__file__).resolve().parent
except NameError:
    _nb_dir = Path.cwd()

for _candidate in [_nb_dir, _nb_dir.parent]:
    _src = _candidate / "src"
    if _src.exists():
        if str(_src) not in sys.path:
            sys.path.insert(0, str(_src))
        break

from rnn_utils import (
    create_neural_targets_from_psth,
    align_behavior_and_neural,
)
from data_io import vectorize_labels
from sweep import generate_search_configs, run_sweep
from sweep_viz import (
    plot_best_per_architecture,
    plot_training_curves_comparison,
    plot_metric_distribution,
    plot_hp_importance,
    plot_hp_heatmap,
    plot_sweep_parallel_coords,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("Using device:", DEVICE)

Using device: mps


## 2. Load Data

### 2a. Behavioral Data

In [2]:
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/Users/lsye/DynamicForagingNPanalysis/data"))
ANIMAL  = "MM012"
SESSION = "20231211_172819"
SESSION_DIR    = DATA_ROOT / ANIMAL / SESSION
TRIALDATA_PATH = SESSION_DIR / "trialdata.csv"

trialdata    = pd.read_csv(TRIALDATA_PATH)
session_data = {"MFblocks": [], "MBblocks": []}

prev_action_left  = vectorize_labels(trialdata, session_data, "prev_choice_left",  rewarded_only=False)
prev_action_right = vectorize_labels(trialdata, session_data, "prev_choice_right", rewarded_only=False)
prev_reward       = vectorize_labels(trialdata, session_data, "prev_reward",        rewarded_only=False)
trial_start       = vectorize_labels(trialdata, session_data, "trial_start",        rewarded_only=False)

y_real = vectorize_labels(trialdata, session_data, "choice", rewarded_only=False).astype(np.int64)

X_real = np.column_stack([
    prev_action_left,
    prev_action_right,
    prev_reward,
    trial_start,
]).astype(np.float32)

X_behav_torch = torch.tensor(X_real, dtype=torch.float32, device=DEVICE).unsqueeze(0)
Y_behav_torch = torch.tensor(y_real, dtype=torch.long,    device=DEVICE).unsqueeze(0)

print(f"Behavioral data: X={tuple(X_behav_torch.shape)}, Y={tuple(Y_behav_torch.shape)}")
print(f"Right-choice fraction: {y_real.mean():.3f}")
print("(Only used if 'behavior' is in RUN_TASKS)")

Behavioral data: X=(1, 337, 4), Y=(1, 337)
Right-choice fraction: 0.531
(Only used if 'behavior' is in RUN_TASKS)


### 2b. Neural Data

In [3]:
neural_targets_dict, neural_meta = create_neural_targets_from_psth(
    animal_name=ANIMAL,
    session=SESSION,
    data_root=str(DATA_ROOT),
    trial_period="choice",
    rewarded_only=False,
    probes=("imec0", "imec1"),
    binwidth_ms=100,
    tpre=1,
    tpost=2,
)

TARGET_KEY = next(iter(neural_targets_dict))
print("Available targets:", list(neural_targets_dict.keys()))
print("Using:", TARGET_KEY, "| shape:", neural_targets_dict[TARGET_KEY].shape)

# Load dmat regressors
_repo = _nb_dir.parent  # repo root (notebooks/ -> repo)
for _c in [_nb_dir.parent, _nb_dir, Path.cwd(), Path.cwd().parent]:
    if (_c / "data" / "dmat-early.npz").exists():
        _repo = _c
        break
DMAT_PATH = _repo / "data" / "dmat-early.npz"

dmat_data = np.load(DMAT_PATH)
X_dmat = dmat_data["X"].astype(np.float32)

n_rows, n_regressors = X_dmat.shape
n_bins_per_trial = 299
n_trials_dmat = n_rows // n_bins_per_trial
n_bins_total = n_trials_dmat * n_bins_per_trial
X_dmat_reshaped = X_dmat[:n_bins_total].reshape(n_trials_dmat, n_bins_per_trial, n_regressors)
X_neural_behavior = X_dmat_reshaped.mean(axis=1).astype(np.float32)

X_neural, Y_neural = align_behavior_and_neural(
    X_neural_behavior, neural_targets_dict[TARGET_KEY]
)

X_neural_torch = torch.tensor(X_neural, dtype=torch.float32, device=DEVICE).unsqueeze(0)
Y_neural_torch = torch.tensor(Y_neural, dtype=torch.float32, device=DEVICE).unsqueeze(0)

print(f"Neural data: X={tuple(X_neural_torch.shape)}, Y={tuple(Y_neural_torch.shape)}")

imec0:ACC excluded 25 unit(s) with mean rate < 1.0 Hz
imec0:DMS excluded 13 unit(s) with mean rate < 1.0 Hz
imec1:M2 excluded 15 unit(s) with mean rate < 1.0 Hz
imec1:DLS excluded 16 unit(s) with mean rate < 1.0 Hz
Available targets: ['imec0_ACC', 'imec0_DMS', 'imec1_M2', 'imec1_DLS']
Using: imec0_ACC | shape: (337, 2280)
Neural data: X=(1, 336, 184), Y=(1, 336, 2280)


## 3. Configure Sweep

Set `RUN_TASKS` to control which tasks are swept.  Use `["neural"]` to run
only the neural prediction sweep, `["behavior"]` for only behavior, or
`["behavior", "neural"]` for both.

Adjust `MAX_CONFIGS` to trade off between thoroughness and runtime.

In [ ]:
RUN_TASKS = ["neural"]  # ["behavior"], ["neural"], or ["behavior", "neural"]

MAX_CONFIGS = 20
PATIENCE = 200
RESULTS_DIR = _nb_dir.parent / "results"
RESULTS_DIR.mkdir(exist_ok=True)
RESULTS_CSV = str(RESULTS_DIR / "sweep_results.csv")

all_configs = []

if "behavior" in RUN_TASKS:
    configs_vanilla_behavior = generate_search_configs("vanilla_rnn", "behavior", max_configs=MAX_CONFIGS, seed=SEED)
    configs_lstm_behavior    = generate_search_configs("lstm",        "behavior", max_configs=MAX_CONFIGS, seed=SEED)
    all_configs += configs_vanilla_behavior + configs_lstm_behavior
    print(f"Vanilla RNN behavior configs: {len(configs_vanilla_behavior)}")
    print(f"LSTM behavior configs:        {len(configs_lstm_behavior)}")
else:
    print("Skipping behavior configs (not in RUN_TASKS)")

if "neural" in RUN_TASKS:
    configs_vanilla_neural = generate_search_configs("vanilla_rnn", "neural", max_configs=MAX_CONFIGS, seed=SEED)
    configs_lstm_neural    = generate_search_configs("lstm",        "neural", max_configs=MAX_CONFIGS, seed=SEED)
    all_configs += configs_vanilla_neural + configs_lstm_neural
    print(f"Vanilla RNN neural configs:   {len(configs_vanilla_neural)}")
    print(f"LSTM neural configs:          {len(configs_lstm_neural)}")
else:
    print("Skipping neural configs (not in RUN_TASKS)")

print(f"Total runs: {len(all_configs)}")

## 4. Run Behavior Sweep

In [ ]:
if "behavior" in RUN_TASKS:
    behavior_configs = configs_vanilla_behavior + configs_lstm_behavior
    df_behavior = run_sweep(
        configs=behavior_configs,
        X_train=X_behav_torch,
        Y_train=Y_behav_torch,
        input_size=X_behav_torch.shape[-1],
        output_size=2,
        device=DEVICE,
        results_csv=RESULTS_CSV,
        patience=PATIENCE,
        print_every=500,
    )
else:
    print("Skipping behavior sweep (not in RUN_TASKS)")

## 5. Run Neural Sweep

In [ ]:
if "neural" in RUN_TASKS:
    neural_configs = configs_vanilla_neural + configs_lstm_neural
    df_neural = run_sweep(
        configs=neural_configs,
        X_train=X_neural_torch,
        Y_train=Y_neural_torch,
        input_size=X_neural_torch.shape[-1],
        output_size=Y_neural_torch.shape[-1],
        device=DEVICE,
        results_csv=RESULTS_CSV,
        patience=PATIENCE,
        print_every=500,
    )
else:
    print("Skipping neural sweep (not in RUN_TASKS)")

## 6. Load and Explore Results

If you've already run the sweep, you can reload results from the CSV.

In [ ]:
df_all = pd.read_csv(RESULTS_CSV)
print(f"Total sweep rows: {len(df_all)}")
print(f"Columns: {list(df_all.columns)}")
df_all.groupby(["model_type", "task_type"])["best_metric"].describe()

## 7. Cross-Architecture Comparisons

### 7a. Best Metric per Architecture

In [ ]:
plot_best_per_architecture(df_all)

### 7b. Training Curves (Best Run per Architecture)

In [ ]:
plot_training_curves_comparison(df_all, task_type="behavior")

In [ ]:
plot_training_curves_comparison(df_all, task_type="neural")

### 7c. Metric Distribution (Box Plot)

In [ ]:
plot_metric_distribution(df_all, task_type="behavior")

In [ ]:
plot_metric_distribution(df_all, task_type="neural")

## 8. Per-Architecture HP Analysis

### 8a. HP Importance (Strip Plots)

In [ ]:
plot_hp_importance(df_all, task_type="behavior")

In [ ]:
plot_hp_importance(df_all, task_type="neural")

### 8b. HP Heatmap (2D interaction)

Explore interactions between two HPs.  Adjust `hp_x` and `hp_y` to investigate
different combinations.

In [ ]:
plot_hp_heatmap(df_all, hp_x="hidden_size", hp_y="lr", task_type="behavior")

In [ ]:
plot_hp_heatmap(df_all, hp_x="hidden_size", hp_y="lr", task_type="neural")

### 8c. Parallel Coordinates

In [ ]:
plot_sweep_parallel_coords(df_all, task_type="behavior", top_n=50)

In [ ]:
plot_sweep_parallel_coords(df_all, task_type="neural", top_n=50)

## 9. Summary

Display the top-5 configs for each task type.

In [ ]:
for tt in df_all["task_type"].unique():
    print(f"\n{'='*80}")
    print(f"Top 5 runs for task_type = {tt}")
    print(f"{'='*80}")
    top5 = df_all[df_all["task_type"] == tt].nlargest(5, "best_metric")
    display_cols = [c for c in top5.columns if c not in ("loss_hist", "metric_hist", "timestamp")]
    display(top5[display_cols])